In [1]:
import sys
import warnings
import numpy as np
import pathlib as pl
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm
from scipy.optimize import minimize
root_folder = pl.Path.cwd().parents[2]
sys.path.insert(0, str(root_folder / "utilities"))
import common_functions as cf

warnings.filterwarnings('ignore')

initial_data_folder = "data/initial_data/function_8"
initial_inputs_path = pl.Path.joinpath(root_folder, initial_data_folder,  "initial_inputs.npy")
initial_outputs_path = pl.Path.joinpath(root_folder, initial_data_folder, "initial_outputs.npy")

In [2]:
data_in = np.load(initial_inputs_path)
data_out = np.load(initial_outputs_path)

Week-01

In [3]:
X_init = data_in
y_init = data_out

bounds = np.array([(0.0, 1.0)] * X_init.shape[1])  # assume normalized [0,1] per dim
dim = X_init.shape[1]
y_best = y_init.max()

kernel = Matern(length_scale=np.ones(dim),
                length_scale_bounds=(1e-2, 1e2),
                nu=2.5)
# alpha = noise variance. If outputs are noisy, increase alpha (e.g. 1e-4 or tuned)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True, n_restarts_optimizer=5, random_state=0)
gp.fit(X_init, y_init)


# -------------------------
# Expected Improvement (EI)
# -------------------------
def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.atleast_2d(x)
    mu, sigma = gp.predict(x, return_std=True)
    sigma = np.maximum(sigma, 1e-12)
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei  # note: maximize this

# Negative EI for minimizers
def neg_ei(x, gp, y_best, xi=0.01):
    return -expected_improvement(x.reshape(1, -1), gp, y_best, xi=xi)[0]

# -------------------------
# Acquisition optimization: many random starts + DE fallback
# -------------------------
def propose_location(gp, y_best, bounds, n_restarts=40):
    dim = bounds.shape[0]
    best_x = None
    best_val = 1e20

    # Random-start L-BFGS-B
    for _ in range(n_restarts):
        x0 = np.random.uniform(bounds[:,0], bounds[:,1])
        res = minimize(fun=neg_ei,
                       x0=x0,
                       args=(gp, y_best),
                       bounds=bounds,
                       method='L-BFGS-B',
                       options={'maxiter':200})
        if res.fun < best_val:
            best_val = res.fun
            best_x = res.x

    # Try differential evolution as a global check (optional, slower)
    try:
        de_res = differential_evolution(lambda x: neg_ei(x, gp, y_best),
                                       bounds=bounds.tolist(),
                                       maxiter=200, polish=True, seed=0)
        if de_res.fun < best_val:
            best_val = de_res.fun
            best_x = de_res.x
    except Exception:
        pass

    # clamp to bounds and return
    best_x = np.clip(best_x, bounds[:,0], bounds[:,1])
    return best_x

# -------------------------
# If you need a small batch (greedy sequential EI)
# -------------------------
def propose_batch(gp, y_best, bounds, batch_size=3):
    batch = []
    gp_copy = gp
    X_aug = X_init.copy()
    y_aug = y_init.copy()
    for i in range(batch_size):
        x_next = propose_location(gp_copy, y_aug.max(), bounds)
        # "Fake" evaluation: use GP predicted mean (Kriging believer)
        y_fake = gp_copy.predict(x_next.reshape(1, -1))[0]
        # Append to augmented set and refit GP (greedy sequential)
        X_aug = np.vstack([X_aug, x_next])
        y_aug = np.hstack([y_aug, y_fake])
        gp_copy = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
        gp_copy.fit(X_aug, y_aug)
        batch.append(x_next)
    return np.array(batch)

# -------------------------
# Run proposal
# -------------------------
x_next = propose_location(gp, y_best, np.array(bounds))
print("Suggested next point (continuous):", x_next)

# If some dims are categorical/encoded, round or map them here, e.g.:
# x_next[cat_index] = int(np.round(x_next[cat_index] * (num_categories-1)))
#
# Example: If dim 6 encodes optimizer 0..3:
# x_next[6] = int(np.round(x_next[6] * 3))

# -------------------------
# (Optional) propose a small batch
# -------------------------
batch = propose_batch(gp, y_best, np.array(bounds), batch_size=3)
print("Suggested batch of 3 points (continuous):\n", batch)


Suggested next point (continuous): [0.20349594 0.16800139 0.19510405 0.03587587 1.         1.
 0.22436766 0.49818666]
Suggested batch of 3 points (continuous):
 [[0.20350236 0.16799864 0.19510626 0.03587702 1.         1.
  0.22436602 0.49868666]
 [0.         0.25829806 0.01137836 0.02416087 1.         1.
  0.         0.49825267]
 [0.17234393 0.         0.29066435 0.         1.         0.
  0.01258338 1.        ]]


In [4]:

# --- Initial data ---
#X_init = np.array([...])  # 30x6 array
#y_init = np.array([...])  # 30 outputs
X_init = data_in
y_init = data_out

y_best = y_init.max()  # best performance so far

# --- Fit Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)

# --- Expected Improvement acquisition function ---
def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu, sigma = mu[0], sigma[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative for minimization in scipy

# --- Optimize acquisition function ---
bounds = [(0,1)]*8
best_x = None
best_ei = float('inf')

for _ in range(20):  # multiple random starts
    x0 = np.random.rand(8)
    res = minimize(lambda x: expected_improvement(x, gp, y_best),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

x_next = best_x
print("Next hyperparameter set to try:", x_next)


Next hyperparameter set to try: [0.08878168 0.28819229 0.04408737 0.25667875 0.63480354 0.45896755
 0.21148179 0.5235234 ]


Week-02

In [5]:
#datain = X_init
#dataout = y_init
# -------------------------
# Existing data (your previous iteration)
# -------------------------
X_init = data_in
y_init = data_out

# Latest evaluated point (the one that gave 9.68029...)
X_new = np.array([0.141081,0.102602,0.176130,0.000000,1.000000,0.409759,0.147143,0.000000])
y_new = np.array([9.895763038795])

# -------------------------
# Step 1: Append new data
# -------------------------
X_all = np.vstack([X_init, X_new])
y_all = np.concatenate([y_init, y_new])

# -------------------------
# Step 2: Fit updated GP surrogate
# -------------------------
bounds = np.array([(0.0, 1.0)] * X_all.shape[1])
dim = X_all.shape[1]

kernel = Matern(length_scale=np.ones(dim),
                length_scale_bounds=(1e-2, 1e2),
                nu=2.5)

# Lower alpha for smooth deterministic behavior
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6,
                              normalize_y=True, n_restarts_optimizer=5,
                              random_state=0)
gp.fit(X_all, y_all)

# -------------------------
# Step 3: Expected Improvement (EI) — exploitative mode
# -------------------------
def expected_improvement(x, gp, y_best, xi=0.001):  # smaller xi = exploit
    x = np.atleast_2d(x)
    mu, sigma = gp.predict(x, return_std=True)
    sigma = np.maximum(sigma, 1e-12)
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei

def neg_ei(x, gp, y_best, xi=0.001):
    return -expected_improvement(x.reshape(1, -1), gp, y_best, xi=xi)[0]

# -------------------------
# Step 4: Optimize acquisition to find next candidate
# -------------------------
def propose_location(gp, y_best, bounds, n_restarts=40):
    best_x, best_val = None, 1e20
    for _ in range(n_restarts):
        x0 = np.random.uniform(bounds[:,0], bounds[:,1])
        res = minimize(fun=neg_ei,
                       x0=x0,
                       args=(gp, y_best, 0.001),
                       bounds=bounds,
                       method='L-BFGS-B',
                       options={'maxiter':200})
        if res.fun < best_val:
            best_val, best_x = res.fun, res.x

    # Optional global search (safety)
    try:
        de_res = differential_evolution(lambda x: neg_ei(x, gp, y_best, 0.001),
                                        bounds=bounds.tolist(),
                                        maxiter=200, polish=True, seed=0)
        if de_res.fun < best_val:
            best_val, best_x = de_res.fun, de_res.x
    except Exception:
        pass

    return np.clip(best_x, bounds[:,0], bounds[:,1])

# Compute updated best observed value
y_best = y_all.max()

# Get next suggested point
x_next = propose_location(gp, y_best, bounds)

print("Next suggested point (exploit mode):", x_next)

Next suggested point (exploit mode): [0.23324259 0.21025667 0.18553569 0.1101389  1.         1.
 0.28427981 1.        ]


Week-03

In [6]:
X_new = np.array([
    [0.141081,0.102602,0.176130,0.000000,1.000000,0.409759,0.147143,0.000000],
    [0.233245,0.210250,0.185536,0.110135,1.000000,1.000000,0.284280,1.000000]
])
y_new = np.array([ 
    9.895763038795,
    9.649813280537
])

# Combine all data
X_all = np.vstack((data_in, X_new))
y_all = np.concatenate((data_out, y_new))


bounds = np.array([(0.0, 1.0)] * X_all.shape[1])
dim = X_all.shape[1]

kernel = Matern(length_scale=np.ones(dim),
                length_scale_bounds=(1e-2, 1e2),
                nu=2.5)

# Lower alpha for smooth deterministic behavior
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6,
                              normalize_y=True, n_restarts_optimizer=5,
                              random_state=0)
gp.fit(X_all, y_all)

# -------------------------
# Step 3: Expected Improvement (EI) — exploitative mode
# -------------------------
def expected_improvement(x, gp, y_best, xi=0.001):  # smaller xi = exploit
    x = np.atleast_2d(x)
    mu, sigma = gp.predict(x, return_std=True)
    sigma = np.maximum(sigma, 1e-12)
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei

def neg_ei(x, gp, y_best, xi=0.001):
    return -expected_improvement(x.reshape(1, -1), gp, y_best, xi=xi)[0]

# -------------------------
# Step 4: Optimize acquisition to find next candidate
# -------------------------
def propose_location(gp, y_best, bounds, n_restarts=40):
    best_x, best_val = None, 1e20
    for _ in range(n_restarts):
        x0 = np.random.uniform(bounds[:,0], bounds[:,1])
        res = minimize(fun=neg_ei,
                       x0=x0,
                       args=(gp, y_best, 0.001),
                       bounds=bounds,
                       method='L-BFGS-B',
                       options={'maxiter':200})
        if res.fun < best_val:
            best_val, best_x = res.fun, res.x

    # Optional global search (safety)
    try:
        de_res = differential_evolution(lambda x: neg_ei(x, gp, y_best, 0.001),
                                        bounds=bounds.tolist(),
                                        maxiter=200, polish=True, seed=0)
        if de_res.fun < best_val:
            best_val, best_x = de_res.fun, de_res.x
    except Exception:
        pass

    return np.clip(best_x, bounds[:,0], bounds[:,1])

# Compute updated best observed value
y_best = y_all.max()

# Get next suggested point
x_next = propose_location(gp, y_best, bounds)

print(cf.format_inputdata(x_next))

0.213214-0.126025-0.227506-0.000000-1.000000-0.330635-0.288952-1.000000


week-04

In [7]:
X_new = np.array([
    [0.141081, 0.102602, 0.176130, 0.000000, 1.000000, 0.409759, 0.147143, 0.000000],
    [0.233245, 0.210250, 0.185536, 0.110135, 1.000000, 1.000000, 0.284280, 1.000000],
    [0.213218, 0.126017, 0.227498, 0.000000, 1.000000, 0.330616, 0.288958, 1.000000]
])
y_new = np.array([ 
    9.895763038795,
    9.649813280537,
    9.842252613667
])

# Combine all data
X_all = np.vstack((data_in, X_new))
y_all = np.concatenate((data_out, y_new))


eps= 1e-20
signs = np.sign(y_all)
signs[signs == 0] = 1.0
y_trans = signs * np.log10(np.abs(y_all) + eps)


kernel = Matern(length_scale = 0.1, nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_all, y_trans)

def acquisition_ei(X, gp, y_best, xi=0.05):
    mu, sigma = gp.predict(X, return_std=True)
    sigma = sigma.reshape(-1, 1)
    mu = mu.reshape(-1, 1)
    imp = mu - y_best - xi
    Z = imp / (sigma + 1e-9)
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei.ravel()

grid_size = 100
margin = 0.03
n = 6

# Create 1D ranges
x1 = np.linspace(margin, 1-margin, n)
x2 = np.linspace(margin, 1-margin, n)
x3 = np.linspace(margin, 1-margin, n)
x4 = np.linspace(margin, 1-margin, n)
x5 = np.linspace(margin, 1-margin, n)
x6 = np.linspace(margin, 1-margin, n)
x7 = np.linspace(margin, 1-margin, n)
x8 = np.linspace(margin, 1-margin, n)

# Create 8D meshgrid
X1, X2, X3, X4, X5, X6, X7, X8 = np.meshgrid(
    x1, x2, x3, x4, x5, x6, x7, x8,
    indexing='ij'
)

# Convert into candidate points
X_candidates = np.vstack([
    X1.ravel(),
    X2.ravel(),
    X3.ravel(),
    X4.ravel(),
    X5.ravel(),
    X6.ravel(),
    X7.ravel(),
    X8.ravel()
]).T

# --- 6. Compute EI across the grid ---
y_best = np.max(y_trans)
acq_values = acquisition_ei(X_candidates, gp, y_best, xi=1.0)

# --- 7. Select the next point ---
next = X_candidates[np.argmax(acq_values)]
best_ei = np.max(acq_values)

# --- 8. Display results with precision ---
#print(f"[{next[0]:.6f}, {next[1]:.6f}, {next[2]:.6f}, {next[3]:.6f}, {next[4]:.6f}, {next[5]:.6f}, {next[6]:.6f}, {next[7]:.6f}]")
print('next output:')
for i in range(8):
    print(f"{next[i]:.6f}")

print(cf.format_inputdata(next))

next output:
0.030000
0.970000
0.030000
0.970000
0.030000
0.970000
0.030000
0.970000
0.030000-0.970000-0.030000-0.970000-0.030000-0.970000-0.030000-0.970000


[0.030000, 0.218000, 0.030000, 0.218000, 0.782000, 0.406000, 0.030000, 0.594000] 0.02 <br/>
[0.030000, 0.406000, 0.030000, 0.218000, 0.594000, 0.406000, 0.030000, 0.594000] 0.05 <br/>
[0.030000, 0.970000, 0.030000, 0.970000, 0.030000, 0.970000, 0.030000, 0.970000] 1.0


Week-05

In [8]:
from sklearn.gaussian_process.kernels import ConstantKernel, WhiteKernel
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
X_new = np.array([
    [0.141081, 0.102602, 0.176130, 0.000000, 1.000000, 0.409759, 0.147143, 0.000000], #week-01 new input
    [0.233245, 0.210250, 0.185536, 0.110135, 1.000000, 1.000000, 0.284280, 1.000000], #week-02 new input
    [0.213218, 0.126017, 0.227498, 0.000000, 1.000000, 0.330616, 0.288958, 1.000000], #week-03 new input
    [0.030000, 0.970000, 0.030000, 0.970000, 0.030000, 0.970000, 0.030000, 0.970000] #week-04 new input
])
y_new = np.array([ 
    9.895763038795, #week-01 output
    9.649813280537, #week-02 output
    9.842252613667, #week-03 output
    8.02656 #week-04 output
])

# Combine all data
X_init = np.vstack((data_in, X_new))
y_init = np.concatenate((data_out, y_new))


eps= 1e-20
signs = np.sign(y_init)
signs[signs == 0] = 1.0
y_trans = signs * np.log10(np.abs(y_init) + eps)

#kernel = Matern(length_scale = 0.1, nu=2.5)
kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=0.25, length_scale_bounds=(1e-2, 2.0), nu=2.5) \
         + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-10, 1e-1))

#gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp = GaussianProcessRegressor(
    kernel=kernel, 
    alpha=1e-8, 
    normalize_y=True,
    n_restarts_optimizer=8,
    random_state=42
    )

gp.fit(X_init, y_trans)

rf = RandomForestRegressor(
    n_estimators=600, 
    random_state=42,
    bootstrap=True,
    max_features=1.0,
    min_samples_leaf=1,
    n_jobs=-1
    )

rf.fit(X_init, y_trans)

gbm_ens=[]
for k in range(12):
    gbm = GradientBoostingRegressor(
        n_estimators=350, 
        learning_rate=0.05, 
        max_depth=3, 
        random_state=100+k,
        subsample=0.8,
        min_samples_leaf=2
        )
    gbm.fit(X_init, y_trans)
    gbm_ens.append(gbm)



def acquisition_ei(X, gp, y_best, xi=0.01):
    mu, sigma = gp.predict(X, return_std=True)
    sigma = sigma.reshape(-1, 1)
    mu = mu.reshape(-1, 1)
    imp = mu - y_best - xi
    Z = imp / (sigma + 1e-9)
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei.ravel()

grid_size = 100
margin = 0.02
n=6
x1 = np.linspace(margin, 1 - margin, n)
x2 = np.linspace(margin, 1 - margin, n)
x3 = np.linspace(margin, 1-margin, n)
x4 = np.linspace(margin, 1-margin, n)
x5 = np.linspace(margin, 1-margin, n)
x6 = np.linspace(margin, 1-margin, n)
x7 = np.linspace(margin, 1-margin, n)
x8 = np.linspace(margin, 1-margin, n)

X1, X2, X3, X4, X5, X6, X7, X8 = np.meshgrid(
    x1, x2, x3, x4, x5, x6, x7, x8, 
    indexing='ij'
)

# Convert into candidate points
X_candidates = np.vstack([
    X1.ravel(),
    X2.ravel(),
    X3.ravel(),
    X4.ravel(),
    X5.ravel(),
    X6.ravel(),
    X7.ravel(),
    X8.ravel()
]).T

# --- 6. Compute EI across the grid ---
y_best = np.max(y_trans)
acq_values = acquisition_ei(X_candidates, gp, y_best, xi=0.5)

# --- 7. Select the next point ---
next_point = X_candidates[np.argmax(acq_values)]
best_ei = np.max(acq_values)

# --- 8. Display results with precision ---
#print(f"Next point using ei: [{next_point[0]:.6f}, {next_point[1]:.6f}, {next_point[2]:.6f}, {next_point[3]:.6f}, {next_point[4]:.6f}, {next_point[5]:.6f}, {next_point[6]:.6f}, {next_point[7]:.6f}]")

print("next output:")
for i in range(8):
    print(f"{next_point[i]:.6f}")

#res_formatted = [f"{r:.6f}" for r in next_point]
#result = "-".join(res_formatted)
print(cf.format_inputdata(next_point))

next output:
0.020000
0.020000
0.020000
0.980000
0.980000
0.980000
0.020000
0.020000
0.020000-0.020000-0.020000-0.980000-0.980000-0.980000-0.020000-0.020000


Week-06

In [ ]:
X_new = np.array([
    [0.141081, 0.102602, 0.176130, 0.000000, 1.000000, 0.409759, 0.147143, 0.000000], #week-01 new input
     [0.233245, 0.210250, 0.185536, 0.110135, 1.000000, 1.000000, 0.284280, 1.000000], #week-02 new input
     [0.213218, 0.126017, 0.227498, 0.000000, 1.000000, 0.330616, 0.288958, 1.000000], #week-03 new input
     [0.030000, 0.970000, 0.030000, 0.970000, 0.030000, 0.970000, 0.030000, 0.970000], #week-04 new input
     [0.020000, 0.020000, 0.020000, 0.980000, 0.980000, 0.980000, 0.020000, 0.020000] #week-05 new input
])

y_new = np.array([ 
    9.895763038795, #week-01 output
     9.649813280537, #week-02 output
     9.842252613667, #week-03 output
     8.02656, #week-04 output
     8.90006 #week-05 output
])

# Combine all data
X_init = np.vstack((data_in, X_new))
y_init = np.concatenate((data_out, y_new))

eps= 1e-20
signs = np.sign(y_init)
signs[signs == 0] = 1.0
y_trans = signs * np.log10(np.abs(y_init) + eps)

#kernel = Matern(length_scale = 0.1, nu=2.5)
kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=0.25, length_scale_bounds=(1e-2, 2.0), nu=2.5) \
         + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-10, 1e-1))

#gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp = GaussianProcessRegressor(
    kernel=kernel, 
    alpha=1e-8, 
    normalize_y=True,
    n_restarts_optimizer=8,
    random_state=42
    )

gp.fit(X_init, y_trans)

rf = RandomForestRegressor(
    n_estimators=600, 
    random_state=42,
    bootstrap=True,
    max_features=1.0,
    min_samples_leaf=1,
    n_jobs=-1
    )

rf.fit(X_init, y_trans)

gbm_ens=[]
for k in range(12):
    gbm = GradientBoostingRegressor(
        n_estimators=350, 
        learning_rate=0.05, 
        max_depth=3, 
        random_state=100+k,
        subsample=0.8,
        min_samples_leaf=2
        )
    gbm.fit(X_init, y_trans)
    gbm_ens.append(gbm)



def acquisition_ei(X, gp, y_best, xi=0.01):
    mu, sigma = gp.predict(X, return_std=True)
    sigma = sigma.reshape(-1, 1)
    mu = mu.reshape(-1, 1)
    imp = mu - y_best - xi
    Z = imp / (sigma + 1e-9)
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei.ravel()

grid_size = 100
margin = 0.02
n=6
x1 = np.linspace(margin, 1 - margin, n)
x2 = np.linspace(margin, 1 - margin, n)
x3 = np.linspace(margin, 1-margin, n)
x4 = np.linspace(margin, 1-margin, n)
x5 = np.linspace(margin, 1-margin, n)
x6 = np.linspace(margin, 1-margin, n)
x7 = np.linspace(margin, 1-margin, n)
x8 = np.linspace(margin, 1-margin, n)

X1, X2, X3, X4, X5, X6, X7, X8 = np.meshgrid(
    x1, x2, x3, x4, x5, x6, x7, x8, 
    indexing='ij'
)

# Convert into candidate points
X_candidates = np.vstack([
    X1.ravel(),
    X2.ravel(),
    X3.ravel(),
    X4.ravel(),
    X5.ravel(),
    X6.ravel(),
    X7.ravel(),
    X8.ravel()
]).T

# --- 6. Compute EI across the grid ---
y_best = np.max(y_trans)
acq_values = acquisition_ei(X_candidates, gp, y_best, xi=0.5)

# --- 7. Select the next point ---
next_point = X_candidates[np.argmax(acq_values)]
best_ei = np.max(acq_values)

# --- 8. Display results with precision ---
#print(f"Next point using ei: [{next_point[0]:.6f}, {next_point[1]:.6f}, {next_point[2]:.6f}, {next_point[3]:.6f}, {next_point[4]:.6f}, {next_point[5]:.6f}, {next_point[6]:.6f}, {next_point[7]:.6f}]")


for i in range(8):
    print(f"{next_point[i]:.6f}")

#res_formatted = [f"{r:.6f}" for r in next_point]
#result = "-".join(res_formatted)
print(cf.format_inputdata(next_point))

0.020000
0.980000
0.020000
0.980000
0.980000
0.020000
0.980000
0.020000
0.020000-0.980000-0.020000-0.980000-0.980000-0.020000-0.980000-0.020000


Week07

In [ ]:
X_new = np.array([
    [0.141081, 0.102602, 0.176130, 0.000000, 1.000000, 0.409759, 0.147143, 0.000000], #week-02 new input
    [0.233245, 0.210250, 0.185536, 0.110135, 1.000000, 1.000000, 0.284280, 1.000000], #week-03 new input
    [0.213218, 0.126017, 0.227498, 0.000000, 1.000000, 0.330616, 0.288958, 1.000000], #week-04 new input
    [0.030000, 0.970000, 0.030000, 0.970000, 0.030000, 0.970000, 0.030000, 0.970000], #week-05 new input
    [0.020000, 0.020000, 0.020000, 0.980000, 0.980000, 0.980000, 0.020000, 0.020000], #week-06 new input
    [0.020000, 0.980000, 0.020000, 0.980000, 0.980000, 0.020000, 0.980000, 0.020000] #week-07 new input
])

y_new = np.array([ 
    9.895763038795, #week-01 output
    9.649813280537, #week-02 output
    9.842252613667, #week-03 output
    8.02656, #week-04 output
    8.90006, #week-05 output
    7.07606 #week-06 output
])

# Combine all data
X_init = np.vstack((data_in, X_new))
y_init = np.concatenate((data_out, y_new))

eps= 1e-20
signs = np.sign(y_init)
signs[signs == 0] = 1.0
y_trans = signs * np.log10(np.abs(y_init) + eps)

#kernel = Matern(length_scale = 0.1, nu=2.5)
kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=0.25, length_scale_bounds=(1e-2, 2.0), nu=2.5) \
         + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-10, 1e-1))

#gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp = GaussianProcessRegressor(
    kernel=kernel, 
    alpha=1e-8, 
    normalize_y=True,
    n_restarts_optimizer=8,
    random_state=42
    )

gp.fit(X_init, y_trans)

rf = RandomForestRegressor(
    n_estimators=600, 
    random_state=42,
    bootstrap=True,
    max_features=1.0,
    min_samples_leaf=1,
    n_jobs=-1
    )

rf.fit(X_init, y_trans)

gbm_ens=[]
for k in range(12):
    gbm = GradientBoostingRegressor(
        n_estimators=350, 
        learning_rate=0.05, 
        max_depth=3, 
        random_state=100+k,
        subsample=0.8,
        min_samples_leaf=2
        )
    gbm.fit(X_init, y_trans)
    gbm_ens.append(gbm)



def acquisition_ei(X, gp, y_best, xi=0.01):
    mu, sigma = gp.predict(X, return_std=True)
    sigma = sigma.reshape(-1, 1)
    mu = mu.reshape(-1, 1)
    imp = mu - y_best - xi
    Z = imp / (sigma + 1e-9)
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei.ravel()

grid_size = 100
margin = 0.02
n=6
x1 = np.linspace(margin, 1 - margin, n)
x2 = np.linspace(margin, 1 - margin, n)
x3 = np.linspace(margin, 1-margin, n)
x4 = np.linspace(margin, 1-margin, n)
x5 = np.linspace(margin, 1-margin, n)
x6 = np.linspace(margin, 1-margin, n)
x7 = np.linspace(margin, 1-margin, n)
x8 = np.linspace(margin, 1-margin, n)

X1, X2, X3, X4, X5, X6, X7, X8 = np.meshgrid(
    x1, x2, x3, x4, x5, x6, x7, x8, 
    indexing='ij'
)

# Convert into candidate points
X_candidates = np.vstack([
    X1.ravel(),
    X2.ravel(),
    X3.ravel(),
    X4.ravel(),
    X5.ravel(),
    X6.ravel(),
    X7.ravel(),
    X8.ravel()
]).T

# --- 6. Compute EI across the grid ---
y_best = np.max(y_trans)
acq_values = acquisition_ei(X_candidates, gp, y_best, xi=0.5)

# --- 7. Select the next point ---
next_point = X_candidates[np.argmax(acq_values)]
best_ei = np.max(acq_values)

# --- 8. Display results with precision ---
print(f"Next point using ei: [{next_point[0]:.6f}, {next_point[1]:.6f}, {next_point[2]:.6f}, {next_point[3]:.6f}, {next_point[4]:.6f}, {next_point[5]:.6f}, {next_point[6]:.6f}, {next_point[7]:.6f}]")
for i in range(8):
    print(f"{next_point[i]:.6f}")

#res_formatted = [f"{r:.6f}" for r in next_point]
#result = "-".join(res_formatted)
print(cf.format_inputdata(next_point))

Next point using ei: [0.980000, 0.020000, 0.020000, 0.980000, 0.980000, 0.020000, 0.020000, 0.020000]
0.980000
0.020000
0.020000
0.980000
0.980000
0.020000
0.020000
0.020000
0.980000-0.020000-0.020000-0.980000-0.980000-0.020000-0.020000-0.020000
